# Baseline semantic search: TF-IDF và BM25

Notebook này xây dựng baseline tìm kiếm phim từ metadata TMDB và dữ liệu MovieLens.

- `LANGUAGE_MODE = "en"` dùng cột tiếng Anh nếu có, sau đó fallback về cột chung.
- Đổi sang `LANGUAGE_MODE = "vi"` khi metadata tiếng Việt đã sẵn sàng.
- TF-IDF và BM25 dùng chung dataframe, tokenizer và hàm `search()`.
- Phần load dữ liệu độc lập với model để có thể thay bằng embedding hoặc collaborative filtering sau này.

In [12]:
# !pip install underthesea

In [13]:
from __future__ import annotations

import json
import re
import ast

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable
from underthesea import word_tokenize


import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [14]:
DATA_DIR = Path("../data")
MOVIELENS_DIR = DATA_DIR / 'movielens'
TMDB_DIR = DATA_DIR / 'tmdb'
ARTIFACT_DIR = DATA_DIR / 'artifacts' / 'lexical_search'

LANGUAGE_MODE = 'vi'  # 'en' hoặc 'vi'
TOP_K = 10
MIN_OVERVIEW_CHARS = 20

if LANGUAGE_MODE not in {'en', 'vi'}:
    raise ValueError("LANGUAGE_MODE phải là 'en' hoặc 'vi'")

print(f'Data directory: {DATA_DIR}')
print(f'Language mode: {LANGUAGE_MODE}')

Data directory: ..\data
Language mode: vi


## 1. Load và chuẩn hóa dữ liệu

TMDB được lưu thành nhiều file JSONL theo batch, vì vậy loader dùng glob và loại bản ghi trùng `movieId`. MovieLens chỉ được đọc những bảng cần cho baseline: `movies.csv`, `links.csv`, `ratings.csv`.

In [15]:
def load_movielens(data_dir: Path) -> dict[str, pd.DataFrame]:
    required = ['movies.csv', 'links.csv', 'ratings.csv']
    missing = [name for name in required if not (data_dir / name).exists()]
    if missing:
        raise FileNotFoundError(f'Thiếu file MovieLens: {missing}')

    return {
        'movies': pd.read_csv(data_dir / 'movies.csv'),
        'links': pd.read_csv(data_dir / 'links.csv'),
        'ratings': pd.read_csv(data_dir / 'ratings.csv'),
    }



def _flatten_genres(value) -> str:
    if pd.isna(value) if not isinstance(value, (list, dict)) else False:
        return ""

    if isinstance(value, str):
        value = value.strip()

        try:
            value = json.loads(value)
        except json.JSONDecodeError:
            try:
                value = ast.literal_eval(value)
            except (ValueError, SyntaxError):
                return ""

    if isinstance(value, list):
        return " ".join(
            item.get("name", "")
            if isinstance(item, dict)
            else str(item)
            for item in value
        ).strip()

    if isinstance(value, dict):
        return str(value.get("name", "")).strip()

    return str(value).replace("|", " ").strip()


def load_tmdb_metadata(tmdb_dir: Path) -> pd.DataFrame:
    metadata = pd.read_csv(TMDB_DIR / 'movies_data_fully_translated.csv', encoding='utf-8-sig')
    metadata.info() 
    
    if 'movieId' not in metadata.columns:
        raise KeyError('TMDB metadata phải có cột movieId để join với MovieLens')

    metadata['movieId'] = pd.to_numeric(metadata['movieId'], errors='coerce')
    metadata = metadata.dropna(subset=['movieId']).drop_duplicates('movieId')
    metadata['movieId'] = metadata['movieId'].astype('int64')
    if 'genres' in metadata:
        metadata['genres_text'] = metadata['genres'].map(_flatten_genres)
    else:
        metadata['genres_text'] = ''
    return metadata


def build_movie_catalog(
    movielens: dict[str, pd.DataFrame],
    tmdb_metadata: pd.DataFrame,
    language_mode: str,
) -> pd.DataFrame:
    if language_mode not in {'en', 'vi'}:
        raise ValueError("language_mode phải là 'en' hoặc 'vi'")

    ratings = movielens['ratings']
    rating_stats = ratings.groupby('movieId', as_index=False).agg(
        rating_count=('rating', 'count'),
        rating_mean=('rating', 'mean'),
    )
    catalog = (
        movielens['movies']
        .merge(movielens['links'], on='movieId', how='left')
        .merge(rating_stats, on='movieId', how='left')
        .merge(tmdb_metadata, on='movieId', how='inner')
    )

    overview_candidates = (['overview_en', 'overview'] if language_mode == 'en'
                          else ['overview_vi', 'overview'])
    title_candidates = (['original_title', 'title'] if language_mode == 'en'
                        else ['title_vi', 'title'])

    def first_available(row: pd.Series, candidates: list[str]) -> str:
        for column in candidates:
            if column in row.index and pd.notna(row[column]) and str(row[column]).strip():
                return str(row[column]).strip()
        return ''

    catalog['search_title'] = catalog.apply(lambda row: first_available(row, title_candidates), axis=1)
    catalog['search_overview'] = catalog.apply(lambda row: first_available(row, overview_candidates), axis=1)
    catalog['search_text'] = (
        catalog['search_title'].fillna('') + ' ' +
        catalog['search_overview'].fillna('') + ' ' +
        catalog['genres_text'].fillna('')
    ).str.replace(r'\s+', ' ', regex=True).str.strip()
    catalog = catalog[catalog['search_overview'].str.len() >= MIN_OVERVIEW_CHARS].copy()
    return catalog.reset_index(drop=True)


movielens = load_movielens(MOVIELENS_DIR)
tmdb_metadata = load_tmdb_metadata(TMDB_DIR)
catalog = build_movie_catalog(movielens, tmdb_metadata, LANGUAGE_MODE)
print(f'MovieLens movies: {len(movielens["movies"]):,}')
print(f'TMDB metadata rows: {len(tmdb_metadata):,}')
print(f'Catalog rows with usable overview: {len(catalog):,}')
display(catalog.head(3))

<class 'pandas.DataFrame'>
RangeIndex: 14824 entries, 0 to 14823
Data columns (total 32 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   tmdbId                 14824 non-null  float64
 1   title_vi               14824 non-null  str    
 2   overview_vi            14824 non-null  str    
 3   genres                 14824 non-null  str    
 4   release_date           14824 non-null  str    
 5   poster_path            14804 non-null  str    
 6   adult                  14824 non-null  bool   
 7   backdrop_path          14412 non-null  str    
 8   belongs_to_collection  2262 non-null   str    
 9   budget                 14824 non-null  int64  
 10  homepage               1 non-null      str    
 11  id                     14824 non-null  int64  
 12  imdb_id                14821 non-null  str    
 13  origin_country         14824 non-null  str    
 14  original_language      14824 non-null  str    
 15  original_titl

,movieId,title_x,genres_x,imdbId,tmdbId_x,rating_count,rating_mean,tmdbId_y,title_vi,overview_vi,...,tagline,title_y,video,vote_average,vote_count,is_ai_translated,genres_text,search_title,search_overview,search_text
0,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602.0,15585.0,3.173981,15602.0,Grumpier Old Men,Một đám cưới gia đình tổ chức mối mối quan hệ ...,...,NaN,Grumpier Old Men,False,6.467,441,True,Phim Lãng Mạn Phim Hài,Grumpier Old Men,Một đám cưới gia đình tổ chức mối mối quan hệ ...,Grumpier Old Men Một đám cưới gia đình tổ chức...
1,4,Waiting to Exhale (1995),Comedy|Drama|Romance,114885,31357.0,2989.0,2.874540,31357.0,Waiting to Exhale,"Bị lừa dối, bị ngược đãi và bị dẫm lên, những ...",...,NaN,Waiting to Exhale,False,6.209,213,True,Phim Hài Phim Chính Kịch Phim Lãng Mạn,Waiting to Exhale,"Bị lừa dối, bị ngược đãi và bị dẫm lên, những ...","Waiting to Exhale Bị lừa dối, bị ngược đãi và ..."
2,5,Father of the Bride Part II (1995),Comedy,113041,11862.0,15474.0,3.077291,11862.0,Father of the Bride Part II,Khi George Banks lấy lại được từ đám cưới của ...,...,NaN,Father of the Bride Part II,False,6.300,837,True,Phim Hài Phim Gia Đình,Father of the Bride Part II,Khi George Banks lấy lại được từ đám cưới của ...,Father of the Bride Part II Khi George Banks l...


## 2. Tokenizer và các model lexical

BM25 được cài đặt tối giản ngay trong notebook để giảm dependency và giữ API tương thích với model khác. Khi production hóa, lớp này có thể chuyển sang `rank_bm25` mà không đổi phần catalog hay hàm tìm kiếm.

In [ ]:
TOKEN_PATTERN = re.compile(r"(?u)\b\w+\b")



def tokenize(text: str, language: str = "vi") -> list[str]:
    if language not in {'en', 'vi'}:
        raise ValueError("language phải là 'en' hoặc 'vi'")
    if language == 'en':
        # Tokenization for English text
        return TOKEN_PATTERN.findall(str(text).lower())
    else:
        # Tokenization for Vietnamese text using underthesea
        return word_tokenize(str(text).lower())


class TfidfSearch:
    def __init__(self) -> None:
        self.vectorizer = TfidfVectorizer(
            tokenizer=tokenize,
            preprocessor=None,
            token_pattern=None,
            sublinear_tf=True,
            min_df=2,
        )
        self.matrix = None

    def fit(self, texts: Iterable[str]) -> 'TfidfSearch':
        self.matrix = self.vectorizer.fit_transform(texts)
        return self

    def scores(self, query: str) -> np.ndarray:
        if self.matrix is None:
            raise RuntimeError('Model chưa được fit')
        query_vector = self.vectorizer.transform([query])
        return linear_kernel(query_vector, self.matrix).ravel()


class Bm25Search:
    def __init__(self, k1: float = 1.5, b: float = 0.75) -> None:
        self.k1 = k1
        self.b = b
        self.documents: list[list[str]] = []
        self.idf: dict[str, float] = {}
        self.doc_lengths = np.array([], dtype=float)
        self.average_doc_length = 0.0

    def fit(self, texts: Iterable[str]) -> 'Bm25Search':
        self.documents = [tokenize(text, language=LAN) for text in texts]
        self.doc_lengths = np.array([len(doc) for doc in self.documents], dtype=float)
        self.average_doc_length = float(self.doc_lengths.mean()) if len(self.doc_lengths) else 0.0
        document_frequency: dict[str, int] = {}
        for document in self.documents:
            for token in set(document):
                document_frequency[token] = document_frequency.get(token, 0) + 1
        document_count = len(self.documents)
        self.idf = {
            token: np.log(1 + (document_count - frequency + 0.5) / (frequency + 0.5))
            for token, frequency in document_frequency.items()
        }
        return self

    def scores(self, query: str) -> np.ndarray:
        if not self.documents:
            raise RuntimeError('Model chưa được fit')
        query_tokens = tokenize(query)
        scores = np.zeros(len(self.documents), dtype=float)
        for index, document in enumerate(self.documents):
            term_counts = pd.Series(document).value_counts()
            denominator_length = self.k1 * (
                1 - self.b + self.b * len(document) / max(self.average_doc_length, 1e-12)
            )
            for token in query_tokens:
                frequency = term_counts.get(token, 0)
                if frequency:
                    scores[index] += self.idf.get(token, 0.0) * (
                        frequency * (self.k1 + 1) / (frequency + denominator_length)
                    )
        return scores

In [17]:
tfidf_model = TfidfSearch().fit(catalog['search_text'])
bm25_model = Bm25Search().fit(catalog['search_text'])


def search(query: str, model_name: str = 'tfidf', top_k: int = TOP_K) -> pd.DataFrame:
    models = {'tfidf': tfidf_model, 'bm25': bm25_model}
    if model_name not in models:
        raise ValueError(f'model_name phải thuộc {sorted(models)}')
    scores = models[model_name].scores(query)
    result = catalog.copy()
    result['score'] = scores
    columns = ['movieId', 'search_title', 'search_overview', 'genres_text', 'rating_count', 'rating_mean', 'score']
    return result.nlargest(top_k, 'score')[columns].reset_index(drop=True)


QUERY = 'space adventure friendship'
display(search(QUERY, model_name='tfidf'))
display(search(QUERY, model_name='bm25'))

NameError: name 'la' is not defined

## 3. Đánh giá nhanh và lưu artifact

Đây là kiểm tra sanity cho baseline, chưa phải đánh giá offline đầy đủ. Khi có tập query/ground truth, có thể thêm Precision@K, Recall@K, MRR hoặc NDCG mà không thay đổi model interface.

In [ ]:
sample_queries = [
    'cuộc phiêu lưu',
    'hài hước lãng mạn',
    'thám tử',
]

for query in sample_queries:
    print(f'\nQuery: {query}')
    print(search(query, model_name='tfidf', top_k=5)[['movieId', 'search_title', 'search_overview', 'score']].to_string(index=False))

In [ ]:
# Lưu catalog và cấu hình để API/service có thể dùng lại sau này.
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
catalog.to_parquet(ARTIFACT_DIR / f'catalog_{LANGUAGE_MODE}.parquet', index=False)
with (ARTIFACT_DIR / f'config_{LANGUAGE_MODE}.json').open('w', encoding='utf-8') as file:
    json.dump(
        {
            'language_mode': LANGUAGE_MODE,
            'text_columns': ['search_title', 'search_overview', 'genres_text'],
            'models': ['tfidf', 'bm25'],
            'tfidf': {'sublinear_tf': True, 'min_df': 2},
            'bm25': {'k1': bm25_model.k1, 'b': bm25_model.b},
            'document_count': len(catalog),
        },
        file,
        ensure_ascii=False,
        indent=2,
    )
print(f'Đã lưu artifacts tại: {ARTIFACT_DIR}')

## 4. Lưu kết quả thí nghiệm và so sánh model

`EVAL_CASES` là tập đánh giá cố định. Mỗi query cần có `relevant_movie_ids`, tức các phim được xem là phù hợp với query. Khi so sánh model chính, giữ nguyên tập query, `top_k` và catalog để kết quả công bằng.

Notebook lưu hai lớp kết quả:

- `experiment_*.json`: cấu hình, metrics tổng hợp và thời gian chạy.
- `query_results_*.csv`: top-k kết quả của từng model cho từng query, dùng để kiểm tra thủ công.

In [ ]:
from datetime import datetime, timezone

# Điền ground truth thật trước khi đánh giá.
# Ví dụ: {'query': 'space adventure', 'relevant_movie_ids': [1, 50, 260]}
EVAL_CASES = []
EVAL_TOP_K = 10


def precision_at_k(retrieved_ids: list[int], relevant_ids: set[int], k: int) -> float:
    retrieved = retrieved_ids[:k]
    return sum(movie_id in relevant_ids for movie_id in retrieved) / max(len(retrieved), 1)


def recall_at_k(retrieved_ids: list[int], relevant_ids: set[int], k: int) -> float:
    if not relevant_ids:
        return 0.0
    return sum(movie_id in relevant_ids for movie_id in retrieved_ids[:k]) / len(relevant_ids)


def reciprocal_rank(retrieved_ids: list[int], relevant_ids: set[int]) -> float:
    for rank, movie_id in enumerate(retrieved_ids, start=1):
        if movie_id in relevant_ids:
            return 1.0 / rank
    return 0.0


def evaluate_model(model_name: str, eval_cases: list[dict], top_k: int = EVAL_TOP_K) -> tuple[dict, pd.DataFrame]:
    query_rows = []
    for case in eval_cases:
        query = case['query']
        relevant_ids = {int(movie_id) for movie_id in case['relevant_movie_ids']}
        results = search(query, model_name=model_name, top_k=top_k)
        retrieved_ids = results['movieId'].astype(int).tolist()
        query_rows.append({
            'model': model_name,
            'query': query,
            'precision_at_k': precision_at_k(retrieved_ids, relevant_ids, top_k),
            'recall_at_k': recall_at_k(retrieved_ids, relevant_ids, top_k),
            'reciprocal_rank': reciprocal_rank(retrieved_ids, relevant_ids),
            'retrieved_movie_ids': retrieved_ids,
        })

    details = pd.DataFrame(query_rows)
    if details.empty:
        return {
            'model': model_name,
            'query_count': 0,
            'precision_at_k': None,
            'recall_at_k': None,
            'mrr': None,
        }, details

    return {
        'model': model_name,
        'query_count': len(details),
        'precision_at_k': float(details['precision_at_k'].mean()),
        'recall_at_k': float(details['recall_at_k'].mean()),
        'mrr': float(details['reciprocal_rank'].mean()),
    }, details


# Model chính sau này chỉ cần có hàm cùng hợp đồng: query -> DataFrame có cột movieId.
def evaluate_custom_model(model_name: str, predict_fn, eval_cases: list[dict], top_k: int = EVAL_TOP_K):
    original_search = globals()['search']
    try:
        globals()['search'] = lambda query, model_name=None, top_k=top_k: predict_fn(query, top_k)
        return evaluate_model(model_name, eval_cases, top_k)
    finally:
        globals()['search'] = original_search


if EVAL_CASES:
    experiment_metrics = []
    experiment_details = []
    for model_name in ['tfidf', 'bm25']:
        metrics, details = evaluate_model(model_name, EVAL_CASES, EVAL_TOP_K)
        experiment_metrics.append(metrics)
        experiment_details.append(details)

    run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    experiment_dir = ARTIFACT_DIR / 'experiments'
    experiment_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = experiment_dir / f'experiment_{run_id}.json'
    details_path = experiment_dir / f'query_results_{run_id}.csv'

    with metrics_path.open('w', encoding='utf-8') as file:
        json.dump({
            'run_id': run_id,
            'language_mode': LANGUAGE_MODE,
            'catalog_rows': len(catalog),
            'top_k': EVAL_TOP_K,
            'evaluation_cases': EVAL_CASES,
            'models': experiment_metrics,
        }, file, ensure_ascii=False, indent=2)
    pd.concat(experiment_details, ignore_index=True).to_csv(details_path, index=False, encoding='utf-8-sig')
    display(pd.DataFrame(experiment_metrics))
    print(f'Metrics: {metrics_path}')
    print(f'Query details: {details_path}')
else:
    print('Chưa có EVAL_CASES. Hãy thêm query và relevant_movie_ids để chạy đánh giá offline.')